In [1]:
import gradio as gr
from openai import OpenAI
import requests

In [2]:
### Checking if Ollama is running
requests.get("http://localhost:11434").content
ollama_url="http://localhost:11434/v1"


In [3]:
###declaring ollama client object 

ollama=OpenAI(base_url= ollama_url, api_key='ollama')

In [4]:
###declaring system prompt
system_prompt="""
You are a customer support triage assistant for a SaaS company.

Your job is to analyze incoming customer messages and return a concise, accurate Json response Object.

Rules:
- Be factual and do not invent details.
- Base your answer only on the customer message.
- Detect the main category from this list:
  ["billing", "technical_issue", "account_access", "feature_request", "refund_request", "general_question", "complaint", "other"]
- Detect priority from this list:
  ["low", "medium", "high", "urgent"]
- Set needs_human to true if the issue involves payment problems, legal threats, data loss, security concerns, repeated frustration, or anything unclear/high-risk.
- Write a short summary in one sentence.
- Draft a professional and empathetic reply in 2 to 4 sentences.
- Keep the reply clear, calm, and helpful.
- Never promise actions that were not explicitly confirmed.
- Output valid JSON only with these keys:
  category, priority, needs_human, summary, draft_reply
"""


In [5]:
def call_ollana(prompt):
    messages =[{'role': 'system', 'content':system_prompt},{'role':'user', 'content':prompt}]
    stream=ollama.chat.completions.create(model="llama3.2",messages=messages, stream=True)

    result=""
    for chunk in stream:
        result+=chunk.choices[0].delta.content or ""
        yield result


In [6]:
inputs = gr.Textbox(
    label="Input text",
    placeholder="Type your question here...",
    info="You can put your text here",
    lines=8
)

outputs = gr.Markdown(label="Model answer")

view = gr.Interface(
    fn=call_ollana,
    title="Welcome to my chat app",
    description="You can ask any question you want",
    inputs=inputs,
    outputs=outputs,
    examples=[
        ["Explain what Einstein's theory is."],
        ["What is the meaning of the word tla9la9?"],
        ["Write a polite email reply."]
    ],
    submit_btn="Send",
    clear_btn="Clear",
    flagging_mode="never"
)

view.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://6c8a9607c41e67f7f6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
